## Imports

In [2]:
# Import all required packages including pyBKT.models.Model!
import numpy as np
import pandas as pd
from pyBKT.models import Model
# import networkx as nx

## Data

In [4]:
student_observations = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')
nodes = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Nodes')
edges = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='KC_Edges')

### Preprocessing

In [5]:
def preprocess(data, kc_col = 'primary_kc_id'):
  # Consider any non-full marks as incorrect (replace 0.5 -> 0)
  data['correct'] = data['score'].case_when([
      (data['score'] == 0.5, 0)
  ]).astype(int)

  # Rename  columns
  obs = data.rename(columns={
      'student_id':     'user_id',
      kc_col:  'skill_name',
      'observation_id': 'order_id'
  })[['user_id', 'skill_name', 'correct', 'order_id']].reset_index(drop=True)
  obs['order_id'] = obs.groupby('user_id').cumcount()
  return obs

## Approach 1 - Full dataset

In [6]:
full_data = preprocess(student_observations)

In [7]:
full_model = Model(seed = 42, num_fits = 1)

In [8]:
full_model.fit(data = full_data)

/Users/mailysguedon/miniforge3/envs/stellar-proj/lib/python3.10/site-packages/pyBKT/fit/M_step.py:61: RuntimeWarning: invalid value encountered in divide
  model['pi_0'] = init_softcounts[:] / np.sum(init_softcounts[:])


In [9]:
params = full_model.params().reset_index()
params

,skill,param,class,value
0,KC.U1.02.observational_unit_variable,prior,default,NaN
1,KC.U1.02.observational_unit_variable,learns,default,1.00000
2,KC.U1.02.observational_unit_variable,guesses,default,0.50000
3,KC.U1.02.observational_unit_variable,slips,default,0.50000
4,KC.U1.02.observational_unit_variable,forgets,default,0.00000
...,...,...,...,...
1175,KC.U10.15.rejection_region_power_calculation,prior,default,NaN
1176,KC.U10.15.rejection_region_power_calculation,learns,default,1.00000
1177,KC.U10.15.rejection_region_power_calculation,guesses,default,0.50000
1178,KC.U10.15.rejection_region_power_calculation,slips,default,0.50000


## Approach 2 - Aggregate KCs

In [ ]:
obs = student_observations.copy()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.00000,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.00000,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.00000,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.00000,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.00000,1,100,B,B,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21803,S025,MF2,27,MF2_FRQ6A,FRQ_Component,FRQ 6,KC.U1.09.quantitative_display_reading,KC.U1.09.quantitative_display_reading|KC.U7.11...,0.00000,1,0,I,Rubric: identify the approximate numeric value...,I
21804,S025,MF2,27,MF2_FRQ6B,FRQ_Component,FRQ 6,KC.U7.12.mean_hypotheses,KC.U7.12.mean_hypotheses|KC.U7.11.matched_pair...,1.00000,1,100,E,Rubric: define the population mean paired diff...,E
21805,S025,MF2,27,MF2_FRQ6C,FRQ_Component,FRQ 6,KC.U7.13.t_test_conditions,KC.U7.13.t_test_conditions|KC.U7.11.matched_pa...,0.50000,1,50,P,"Rubric: address paired structure, random sampl...",P
21806,S025,MF2,27,MF2_FRQ6D,FRQ_Component,FRQ 6,KC.U7.14.t_test_statistic_mean,KC.U7.14.t_test_statistic_mean|KC.U7.15.p_valu...,1.00000,1,100,E,"Rubric: use mean difference, standard error, d...",E


In [ ]:
import pandas as pd
G = nx.DiGraph()
G.add_nodes_from(nodes["kc_id"])

# Only use prerequisite edges to define hierarchy
G.add_edges_from(zip(edges["source_kc_id"], edges["target_kc_id"]))

# Find root nodes (no predecessors = top of the chain)
roots = [n for n in G.nodes if G.in_degree(n) == 0]
roots
# Assign each KC to its earliest ancestor (root)
kc_to_group = {}
for root in roots:
    descendants = nx.descendants(G, root) | {root}
    for kc in descendants:
        kc_to_group[kc] = root  # or overwrite with most recent ancestor

nodes["coarse_kc"] = nodes["kc_id"].map(kc_to_group)

In [ ]:
# Create a simple lookup: fine KC -> coarse KC
kc_mapping = nodes[["kc_id", "coarse_kc"]].set_index("kc_id")["coarse_kc"].to_dict()

# Map primary_kc_id to its coarse KC
obs["coarse_kc"] = obs["primary_kc_id"].map(kc_mapping)

# Count observations per coarse KC
obs_counts = obs.groupby("coarse_kc").agg(
    n_observations=("observation_id", "count"),
    n_students=("student_id", "nunique"),
).reset_index()

obs_counts.head()


,coarse_kc,n_observations,n_students
0,KC.U1.01.statistical_context,2556,25
1,KC.U1.12.center_mean_median_mode,74,25
2,KC.U1.13.spread_iqr_range_sd,667,25
3,KC.U1.18.linear_transformations,1037,25
4,KC.U2.08.explanatory_response_variables,1726,25


In [ ]:
agg_data = preprocess(obs,'coarse_kc' )

In [ ]:
agg_model = Model(seed = 42, num_fits = 1)

In [ ]:
agg_model.fit(data = agg_data)

In [ ]:
params = agg_model.params().reset_index()
params

,skill,param,class,value
0,KC.U1.01.statistical_context,prior,default,0.66526
1,KC.U1.01.statistical_context,learns,default,0.00002
2,KC.U1.01.statistical_context,guesses,default,0.40988
3,KC.U1.01.statistical_context,slips,default,0.36587
4,KC.U1.01.statistical_context,forgets,default,0.00000
...,...,...,...,...
190,KC.U9.12.slope_interval_width_sample_size,prior,default,0.91080
191,KC.U9.12.slope_interval_width_sample_size,learns,default,0.29290
192,KC.U9.12.slope_interval_width_sample_size,guesses,default,0.13881
193,KC.U9.12.slope_interval_width_sample_size,slips,default,0.38003


## Evaluation

In [ ]:
training_rmse = full_model.evaluate(data = full_data)
training_auc = full_model.evaluate(data = full_data, metric = 'auc')
print("Training RMSE: %f" % training_rmse)
print("Training AUC: %f" % training_auc)

Training RMSE: 0.488175
Training AUC: 0.622438


In [ ]:
training_rmse = agg_model.evaluate(data = agg_data)
training_auc = agg_model.evaluate(data = agg_data, metric = 'auc')
print("Training RMSE: %f" % training_rmse)
print("Training AUC: %f" % training_auc)

Training RMSE: 0.487972
Training AUC: 0.621735
